In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Optimized for Colab T4/L4/A100
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.57.3
!pip install --no-deps trl==0.22.2

In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, standardize_data_formats

In [ ]:
max_seq_length = 16000 # Supporting long-context coding tasks

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/LFM2.5-1.2B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # Saves VRAM for Colab
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "out_proj", "in_proj", "w1", "w2", "w3"],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # 30% less VRAM usage
    random_state = 3407,
)

==((====))==  Unsloth 2026.3.4: Fast Lfm2 patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Making `model.base_model.model.model` require gradients


In [ ]:
dataset = load_dataset("ise-uiuc/Magicoder-Evol-Instruct-110K", split = "train[:15000]")

# Apply ChatML template to tokenizer
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
)

def formatting_prompts_func(examples):
    # Safe extraction of keys to avoid KeyError
    instructions = examples.get("instruction", [])
    inputs       = examples.get("input", [""] * len(instructions)) # Default to empty string if missing
    outputs      = examples.get("response", [])

    texts = []
    for instr, inp, out in zip(instructions, inputs, outputs):
        # Combine instruction and input if input exists
        full_user_msg = f"{instr}\n{inp}".strip() if inp else instr

        convo = [
            {"role": "user", "content": full_user_msg},
            {"role": "assistant", "content": out},
        ]

        # Apply template and clean up
        formatted_text = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        # Remove the BOS token if it's prepended (common in LFM/Qwen models)
        if tokenizer.bos_token and formatted_text.startswith(tokenizer.bos_token):
            formatted_text = formatted_text.replace(tokenizer.bos_token, "")

        texts.append(formatted_text)

    return { "text" : texts }

# Map the dataset
dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        # max_steps = 60,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",

        # --- ADD THESE TWO LINES TO REMOVE ACCOUNT ATTACHMENTS ---
        report_to = "none",     # Disables wandb/comet/etc.
        push_to_hub = False,    # Disables Hugging Face account syncing
        # --------------------------------------------------------
    ),
)

# Execute training
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 15,000 | Num Epochs = 1 | Total steps = 1,875
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 22,216,704 of 1,192,557,312 (1.86% trained)


Step,Training Loss
1,1.756300
2,1.599000
3,1.679200
4,1.829100
5,1.772100
6,1.284800
7,2.339800
8,1.315100
9,1.615800
10,1.603400


In [ ]:
# # 1. Prepare the model for faster inference
# FastLanguageModel.for_inference(model)

# # 2. Define your coding prompt
# messages = [
#     {"role": "user", "content": "Write a Python script to find the shortest path in a graph using Dijkstra's algorithm."},
# ]

# # 3. Tokenize with attention_mask
# inputs = tokenizer.apply_chat_template(
#     messages,
#     tokenize = True,
#     add_generation_prompt = True,
#     return_tensors = "pt",
#     return_dict = True, # This ensures both input_ids and attention_mask are returned
# ).to("cuda")

# # 4. Generate by passing the unpacked dictionary (**inputs)
# outputs = model.generate(
#     **inputs,            # This automatically passes input_ids AND attention_mask
#     max_new_tokens = 512,
#     use_cache = True,
#     pad_token_id = tokenizer.pad_token_id # Explicitly set the pad token ID
# )

# # 5. Decode and print
# print(tokenizer.batch_decode(outputs))


In [ ]:
from transformers import TextStreamer

# 1. Prepare the model for faster inference
FastLanguageModel.for_inference(model)

# 2. Define your coding prompt
messages = [
    {"role": "user", "content": "Write a Python script to find the shortest path in a graph using Dijkstra's algorithm."},
]

# 3. Setup the real-time streamer
text_streamer = TextStreamer(tokenizer)

# 4. Tokenize with attention_mask
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
).to("cuda")

# 5. Generate with streaming enabled
print("Generating Response...\n" + "="*20)
_ = model.generate(
    **inputs,
    streamer = text_streamer,    # This shows the code as it's being typed
    max_new_tokens = 512,
    use_cache = True,
    pad_token_id = tokenizer.pad_token_id,

    # Coding-specific optimizations
    temperature = 0.2,           # Lower temperature = more precise code
    repetition_penalty = 1.1,    # Prevents the model from repeating same lines
)

Generating Response...
<|im_start|>user
Write a Python script to find the shortest path in a graph using Dijkstra's algorithm.<|im_end|>
<|im_start|>assistant
Here is a Python script that uses Dijkstra's algorithm to find the shortest path in a graph:

```python
import heapq

def 

In [ ]:
model.save_pretrained("lora_model2_coding")
tokenizer.save_pretrained("lora_model2_coding")

('lora_model2_coding/tokenizer_config.json',
 'lora_model2_coding/special_tokens_map.json',
 'lora_model2_coding/chat_template.jinja',
 'lora_model2_coding/tokenizer.json')

In [ ]:
# max configs max step =1,
# seq length 16k
# increse datset 50k,


In [ ]:
# !zip -r lora_model_coding.zip lora_model_coding

In [ ]:
# from google.colab import files
# files.download('lora_model_coding.zip')